## Install & Import

In [ ]:
import pandas as pd
import numpy as np
import json
import os
import time
import matplotlib.pyplot as plt
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("USE DATABASE SNOWFLAKE_LEARNING_DB").collect()
session.sql("USE SCHEMA PUBLIC").collect()
STAGE = "@SNOWFLAKE_LEARNING_DB.PUBLIC.EXTERNAL_DATA_STAGE"

TARGET_COLS = [
    'Electrical Conductance',
    'Total Alkalinity',
    'Dissolved Reactive Phosphorus'
]
TARGET_SHORT = {
    'Electrical Conductance': 'ec',
    'Total Alkalinity': 'alkalinity',
    'Dissolved Reactive Phosphorus': 'drp'
}

# Load data
for f in ['master_train.parquet', 'master_test.parquet', 'selected_features.json']:
    if not os.path.exists(f'/tmp/{f}'):
        session.file.get(f"{STAGE}/{f}", "/tmp/")

master_train = pd.read_parquet('/tmp/master_train.parquet')
master_test = pd.read_parquet('/tmp/master_test.parquet')

with open('/tmp/selected_features.json') as f:
    feat_selection = json.load(f)

print(f"✓ Train: {master_train.shape}, Test: {master_test.shape}")
for short, info in feat_selection['selected_features'].items():
    print(f"  {short}: {info['n_selected']} features (CV R²={info['best_cv_r2']})")

## Train

In [ ]:
import joblib
region_bundle = joblib.load('/tmp/region_groups.joblib')
region_array = region_bundle['region_array']

def train_oof_single_target(train_df, test_df, target_col, feature_cols,
                             groups, n_splits=4, n_estimators=200, max_depth=55,
                             use_log_transform=True, random_state=42):
    """
    10-Fold Out-of-Fold training untuk 1 target.
    
    Mengikuti pemenang 2025:
    - ETR(n_estimators=200, max_depth=55) → mereka pakai parameter ini
    - OOF: prediksi pada fold yang TIDAK dilihat saat training
    - Log-transform target: di dalam fold, fit on train fold only
    - Imputasi & scaling: di dalam fold juga (anti data leakage)
    
    Returns: (oof_predictions, test_predictions, fold_scores, models)
    """
    short = TARGET_SHORT[target_col]
    print(f"\n{'='*60}")
    print(f"TRAINING: {short} ({target_col})")
    print(f"  Features: {len(feature_cols)}, Estimators: {n_estimators}, MaxDepth: {max_depth}")
    print(f"{'='*60}")
    
    X = train_df[feature_cols].values
    y = train_df[target_col].values
    X_test = test_df[feature_cols].values
    
    # Check skewness untuk log-transform decision
    skew = pd.Series(y).dropna().skew()
    do_log = use_log_transform and abs(skew) > 1
    if do_log:
        print(f"  ⚠ Target skew={skew:.2f} → log1p transform enabled")
    
    n_test = X_test.shape[0]
    oof_preds = np.zeros(len(X))
    test_preds_sum = np.zeros(n_test)
    fold_scores = []
    models = []
    
    kf = GroupKFold(n_splits=n_splits)
    
    for fold_idx, (train_idx, val_idx) in enumerate(kf.split(X, y, groups), 1):
        X_tr, X_val = X[train_idx], X[val_idx]
        y_tr, y_val = y[train_idx], y[val_idx]
        
        # Log-transform target (di dalam fold!)
        if do_log:
            y_tr_fit = np.log1p(y_tr)
        else:
            y_tr_fit = y_tr.copy()
        
        # Imputasi dalam fold: replace NaN/inf dengan 0
        X_tr = np.nan_to_num(X_tr, nan=0.0, posinf=0.0, neginf=0.0)
        X_val = np.nan_to_num(X_val, nan=0.0, posinf=0.0, neginf=0.0)
        X_test_clean = np.nan_to_num(X_test, nan=0.0, posinf=0.0, neginf=0.0)
        
        model = ExtraTreesRegressor(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=random_state + fold_idx,
            n_jobs=-1
        )
        model.fit(X_tr, y_tr_fit)
        
        # Predict validation fold
        val_pred = model.predict(X_val)
        if do_log:
            val_pred = np.expm1(val_pred)  # Inverse log1p
        oof_preds[val_idx] = val_pred
        
        # Predict test (accumulate, average later)
        test_pred = model.predict(X_test_clean)
        if do_log:
            test_pred = np.expm1(test_pred)
        test_preds_sum += test_pred
        
        # Score
        fold_r2 = r2_score(y_val, val_pred)
        fold_scores.append(fold_r2)
        models.append(model)
        
        print(f"  Fold {fold_idx}/{n_splits}: R²={fold_r2:.4f}")
    
    test_preds_avg = test_preds_sum / n_splits
    
    mean_r2 = np.mean(fold_scores)
    std_r2 = np.std(fold_scores)
    print(f"\n  ✓ Mean CV R²: {mean_r2:.4f} ± {std_r2:.4f}")
    
    return oof_preds, test_preds_avg, fold_scores, models

print("✓ Training engine ready")

In [ ]:
from sklearn.model_selection import GroupKFold
all_results = {}
all_test_preds = {}
t_start = time.time()

for target_col in TARGET_COLS:
    short = TARGET_SHORT[target_col]
    
    # Ambil selected features dari RFECV
    feat_info = feat_selection['selected_features'][short]
    feature_cols = feat_info['features']
    
    # Pastikan semua feature ada di master data
    available = [c for c in feature_cols if c in master_train.columns and c in master_test.columns]
    missing = set(feature_cols) - set(available)
    if missing:
        print(f"  ⚠ {short}: {len(missing)} features missing, using {len(available)} available")
    
    oof_preds, test_preds, fold_scores, models = train_oof_single_target(
        train_df=master_train,
        test_df=master_test,
        target_col=target_col,
        feature_cols=available,
        groups=region_array, 
        n_splits=4,
        n_estimators=200,
        max_depth=55,
        use_log_transform=True,
        random_state=42
    )
    
    all_results[short] = {
        'oof_preds': oof_preds,
        'fold_scores': fold_scores,
        'models': models,
        'features_used': available
    }
    all_test_preds[target_col] = test_preds

elapsed = time.time() - t_start
print(f"\n{'='*60}")
print(f"TRAINING COMPLETE — {elapsed:.0f}s ({elapsed/60:.1f} min)")
print("=" * 60)

In [ ]:
print("FINAL SCORES:")
r2_scores = []
for target_col in TARGET_COLS:
    short = TARGET_SHORT[target_col]
    scores = all_results[short]['fold_scores']
    mean_r2 = np.mean(scores)
    r2_scores.append(mean_r2)
    print(f"  {short}: Mean R² = {mean_r2:.4f} ± {np.std(scores):.4f}")

overall_mean_r2 = np.mean(r2_scores)
print(f"\n  ★ MEAN R² (leaderboard metric): {overall_mean_r2:.4f}")

# 5b. Feature importance per target
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for i, target_col in enumerate(TARGET_COLS):
    short = TARGET_SHORT[target_col]
    models = all_results[short]['models']
    features = all_results[short]['features_used']
    
    # Average importance across all fold models
    importances = np.mean([m.feature_importances_ for m in models], axis=0)
    top_idx = np.argsort(importances)[-15:]  # Top 15
    
    axes[i].barh(range(len(top_idx)), importances[top_idx], color=['#2196F3', '#4CAF50', '#FF9800'][i])
    axes[i].set_yticks(range(len(top_idx)))
    axes[i].set_yticklabels([features[j][:30] for j in top_idx], fontsize=8)
    axes[i].set_title(f'{short}\nR²={np.mean(all_results[short]["fold_scores"]):.4f}')
    axes[i].set_xlabel('Importance')

plt.suptitle('Top 15 Feature Importance per Target', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## Predict

In [ ]:
local_template = "/tmp/submission_template.csv"
if not os.path.exists(local_template):
    session.file.get(f"{STAGE}/submission_template.csv", "/tmp/")
submission = pd.read_csv(local_template)

# Isi prediksi
for target_col in TARGET_COLS:
    submission[target_col] = all_test_preds[target_col]

# Validasi: pastikan tidak ada NaN
assert submission[TARGET_COLS].isnull().sum().sum() == 0, "Ada NaN di prediksi!"

# Hapus kolom yang tidak diperlukan untuk submission
keep_cols = ['Latitude', 'Longitude', 'Sample Date'] + TARGET_COLS
submission = submission[[c for c in keep_cols if c in submission.columns]]

# Simpan & upload
submission.to_csv('/tmp/submission.csv', index=False)
session.file.put("file:///tmp/submission.csv", STAGE, auto_compress=False, overwrite=True)

print(f"✓ Submission saved: {submission.shape}")
print(f"\nSample predictions:")
print(submission[TARGET_COLS].describe().round(2))

# Sanity check: bandingkan distribusi prediksi vs training
print("\n--- Distribution Comparison ---")
for col in TARGET_COLS:
    short = TARGET_SHORT[col]
    train_median = master_train[col].median()
    pred_median = submission[col].median()
    print(f"  {short}: train median={train_median:.1f}, pred median={pred_median:.1f}")

## Save Artifact

In [ ]:
import pickle

# Simpan semua model + metadata
artifacts = {
    'target_cols': TARGET_COLS,
    'overall_mean_r2': overall_mean_r2,
}

for target_col in TARGET_COLS:
    short = TARGET_SHORT[target_col]
    artifacts[f'{short}_fold_scores'] = all_results[short]['fold_scores']
    artifacts[f'{short}_features'] = all_results[short]['features_used']
    artifacts[f'{short}_n_models'] = len(all_results[short]['models'])

# Simpan metadata (tanpa model objects — terlalu besar)
with open('/tmp/model_artifacts.json', 'w') as f:
    json.dump(artifacts, f, indent=2, default=str)
session.file.put("file:///tmp/model_artifacts.json", STAGE, auto_compress=False, overwrite=True)

# Simpan model objects via pickle
for target_col in TARGET_COLS:
    short = TARGET_SHORT[target_col]
    models = all_results[short]['models']
    with open(f'/tmp/models_{short}.pkl', 'wb') as f:
        pickle.dump(models, f)
    session.file.put(f"file:///tmp/models_{short}.pkl", STAGE, auto_compress=False, overwrite=True)

print(f"✓ All artifacts saved")
print(f"  ★ Leaderboard metric (Mean R²): {overall_mean_r2:.4f}")